Bronze Delta Table(10 incoming records from 2 CSV batches) -> PySpark transformations(Cast data types, validate records, identity latest versions) -> Window functions(Choose one latest record per order ID) -> Delta MERGE (Update existing orders and insert new ones) -> Silver Delta table(8 unique orders with their latest statuses)

Upsert combines two operations:
    Update: If the order already exists, update its information when incoming version is newer
    Insert: If order doesn't exist, insert it

functions as F which is PySpark functions access column operations, casting, and validation
from...import Window which is Window functions rank records within each order group
from delta.tables import DeltaTable which is Delta Lake API execute MERGE operations
bronze_table =... which is source configuration identifies our raw input
silver_table=... which is target configuration identifies our cleaned current-state output

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

catalog="workspace"
schema = "incremental_project"

bronze_table = f"{catalog}.{schema}.orders_bronze_v3"
silver_table = f"{catalog}.{schema}.orders_silver_step4"

print("Bronze source:", bronze_table)
print("Silver destination:", silver_table)

In [ ]:
assert spark.catalog.tableExists(bronze_table), (
    f"Missing Bronze table: {bronze_table}"
)

print("Cell 1 passed: Bronze source exists")

spark.table(bronze_table) creates a DataFrame representing the existing Bronze table
bronze_df.select(...) chooses columns to display
.orderBy("order_id","updated_at") sorts the displayed data by order ID and source update time
Spark generally evaluates DataFrame transformations lazily. Creating or transforming a DataFrame defines a computation; actions cause Spark to execute it
bronze_df.count() is an action that computes the record count. display(...) also causes Databricks to evaluate the DataFrame for presentation

In [ ]:
bronze_df = spark.table(bronze_table)

display(
    bronze_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_amount",
        "updated_at",
        "_source_file"
    ).orderBy("order_id","updated_at")
)

print("Bronze rows:", bronze_df.count())

In [ ]:
assert bronze_df.count() == 10
assert(
    bronze_df.select("order_id").distinct().count() == 8
)

print("Cell 2 passed: 10 raw rows, 8 unique orders")

F.col("order_amount").cast("decimal(12,2)") converts incoming order amounts into decimal values with precision 12 and scale 2. For money related values, decimal types avoid many vinary floating point precision issues
F.to_timestamp("updated_at") parses the source timestamp string into a timestamp value Spark can sort and compare as a time value. We need this because later we'll choose the most recetnly updated order
.filter(
    F.col("order_id").isNotNull()
    & F.col("order_amount").isNotNull()
    & (F.col("order_amount") >= 0)
) keeps rows meeting selected validation rules. & operator combines Boolean expressions, similar to SQL's AND. Filtering records does not preserve rejected rows. 

In [ ]:
silver_input = (
    bronze_df
    .withColumn(
        "order_amount",
        F.col("order_amount").cast("decimal(12,2)")
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("order_status").isNotNull()
        & F.col("order_amount").isNotNull()
        & (F.col("order_amount") >= 0)
        & F.col("updated_at").isNotNull()
        & F.col("_rescued_data").isNull()
    )
)

silver_input.printSchema()

In [ ]:
print("Cleaned rows:", silver_input.count())

assert silver_input.count() == 10

silver_input.select(
    "order_id", "order_amount", "updated_at"
).printSchema()

print("Cell 3 passed: All 10 records are valid")

A window function calculates a result across a group of related rows without necessarily collapsing those rows into one aggregate

Window.partitionBy("order_id") Groups rows logically by order_id for window calculation. This is not the same as writing files into physically partitioned Delta directories

F.col("updated_at").desc() places the newest timestamp first

F.row_number().over(order_window) assigns sequential numbers beginning with 1 inside each order_id group

F.col("_ingested_at").desc()
F.col("_source_file").desc() provides secondary ordering if update timestamps match. This is suitable as a practice rule, but a real CDC feed should ideally supply an authoritative change sequence number. Arrival time and filename may not reflect the true order of business changes

In [ ]:
order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_ingested_at").desc(),
        F.col("_source_file").desc()
    )
)

ranked_df = silver_input.withColumn(
    "row_num",
    F.row_number().over(order_window)
)

display(
    ranked_df
    .filter(F.col("order_id").isin("ORD002", "ORD003"))
    .select(
        "order_id",
        "order_status",
        "updated_at",
        "row_num"
    )
    .orderBy("order_id", "row_num")
)

.filter(F.col("row_num")==1) keeps one selected version per order ID compare that with bronze_df.dropDuplicates() can remove repeated order IDs, but it does not guarantee that retained record is latest one. That's why we're using row_number() with an explicit ordering rule 

In [ ]:
latest_orders = (
    ranked_df
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

display(latest_orders.orderBy("order_id"))

In [ ]:
assert latest_orders.count() == 8

assert (
    latest_orders.select("order_id").distinct().count() == 8
)

display(
    latest_orders
    .filter(F.col("order_id").isin("ORD002","ORD003"))
    .select("order_id", "order_status")
)

print("Cell 5 passed: 8 latest orders")

We could write all eight latest records directly into Silver, but then you wouldn't get to observe a meaningful MERGE

Instead, we'll simulate real production flow:
- Batch 001 initializes the Silver table
- Batch 002 applies three inserts and two updates

F.col("_source_file").endswith("/orders_batch_002.csv") uses the _source_file metadata column added in Step 3 to identify which incoming batch produced each record

In production, file metadata alone is generally not a complete CDC mechanism. We'd usually track new source events using a checkpoint, a processed-file log, or an explicit change feed

In [ ]:
batch_001 = silver_input.filter(
    F.col("_source_file").endswith(
        "/orders_batch_001.csv"
    )
)

batch_002 = silver_input.filter(
    F.col("_source_file").endswith(
        "/orders_batch_002.csv"
    )
)

print("Batch 001:", batch_001.count())
print("Batch 002:", batch_002.count())

assert batch_001.count() == 5
assert batch_002.count() == 5

In [ ]:
silver_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_amount",
    "updated_at",
    "_ingested_at",
    "_source_file"
]


spark.catalog.tableExists() is safe initialization prevents accidental reinitialization
.select(*silver_columns) is schema projection selects the fields that belong in Silver
.format("delta") is delta lake storage that stores the table in Delta format
.mode("errorifexists") is write protection that fails rather than overwriting an existing table
.saveAsTable(silver_table) is table creation that persists DataFrame as a named table

In [ ]:
silver_table = 'your_silver_table_name'
assert not spark.catalog.tableExists(silver_table), (
    "The step 4 Silver table already exits."
    "Inspect it before attempting initialization again."
)
(
    batch_001
    .select(*silver_columns)
    .write
    .format("delta")
    .mode("error")
    .saveAsTable(silver_table)
)

print("Initial Silver table created")

In [ ]:
initial_silver = spark.table(silver_table)

assert initial_silver.count() == 5

display(initial_silver.orderBy("order_id"))

Before MERGE, we must ensure source has at most one record for each order ID because multiple source records trying to update the same target order can make a MERGE ambiguous. Databricks recommends deduplicating source before MERGE
Group changes by business key is implemented at Window.partitionBy("order_id")
Find newest source version is implemented at orderBy(F.col("updated_at").desc())
Assign version rank is implemented at F.row_number().over(...)
Keep only latest version is implemented at .filter(F.col("row_num")==1)
Match target schema is implemented at .select(*silver_columns)

In [ ]:
batch_002_window = (
    Window.partitionBy("order_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_ingested_at").desc(),
        F.col("_source_file").desc()
    )
)

merge_source = (
    batch_002
    .withColumn(
        "row_num",
        F.row_number().over(batch_002_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .select(*silver_columns)
)

assert merge_source.count() == 5

assert(
    merge_source.select("order_id").distinct().count() == 5
)

display(merge_source.orderBy("order_id"))

target = DeltaTable.forName(spark, silver_table) retrieves a DeltaTable object so we can perform transactional operations
.merge(merge_source.alias("source"),"target.order_id = source.order_id") matches incoming orders with existing Silver orders using order_id. In SQL, this is similar to an ON condition in a join
.whenMatchedUpdateAll(
    condition="source.updated_at > target.updated_at"
) updates an existing order when its incoming timestamp is newer


In [ ]:
target = DeltaTable.forName(
    spark,
    silver_table
)
(
    target.alias("target")
    .merge(
        merge_source.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed successfully!")

In [ ]:
silver_df = spark.table(silver_table)

total_rows = silver_df.count()

unique_orders = (
    silver_df.select("order_id")
    .distinct()
    .count()
)

print("Silver rows:", total_rows)
print("Unique orders:", unique_orders)

display(
    silver_df
    .select(
        "order_id",
        "order_status",
        "order_amount",
        "updated_at"
    )
    .orderBy("order_id")
)

In [ ]:
assert total_rows == 8
assert unique_orders == 8

statuses = {
    row["order_id"]: row["order_status"]
    for row in silver_df
        .filter(F.col("order_id").isin("ORD002", "ORD003"))
        .select("order_id", "order_status")
        .collect()
}

assert statuses["ORD002"] == "delivered"
assert statuses["ORD003"] == "shipped"

assert dict(silver_df.dtypes)["order_amount"] == "decimal(12,2)"
assert dict(silver_df.dtypes)["updated_at"] == "timestamp"

print("Step 4 Silver validation passed!")

In [ ]:
silver_after_rerun = spark.table(silver_table)

assert silver_after_rerun.count() == 8

assert (
    silver_after_rerun.select("order_id").distinct().count() == 8
)

print("MERGE rerun validation passed!")

Bronze keeps incoming raw records, source values retained as strings, and auto loader ingestion. Silver maintains cleaned current records, important columns converted to appropriate types, window functions + Delta merge

We use ROW_NUMBER() with PARTITION BY order_id and ORDER BY updated_at DESC to reliably select latest version of each order. dropDuplicates() doesn't guarantee which duplicate record is retained. If two records have same updated_at, we need another reliable field to break the tie
Auto loader checkpointing prevents repeated ingestion of previously processed files by tracking stream progress. MERGE prevents duplicate current-state records by matching business keys and applies updates only when incoming data is newer